In [2]:
from abc import ABC, abstractmethod
from pathlib import Path
from typing import List

from langchain_core.documents import Document
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader
)

In [3]:
class BaseDocumentLoader(ABC):
    """Common interface and metadata handling for document loaders."""

    supported_extensions: set[str] = set()

    def load(self, file_path: str | Path) -> List[Document]:
        path = Path(file_path)

        if not path.exists():
            raise FileNotFoundError(f"Document not found: {path}")

        if not path.is_file():
            raise ValueError(f"Expected a file, received {path}")

        if path.suffix.lower() not in self.supported_extensions:
            raise ValueError(
                f"Unsupported file type: {path.suffix} "
                f"Supported types: {sorted(self.supported_extensions)}."
            )

        documents = self._load(path)

        if not documents:
            raise ValueError(f"No text extracted from document: {path}")

        for document in documents:
            document.metadata["source"] = str(path)
            document.metadata["document_id"] = path.stem
            document.metadata["file_type"] = path.suffix.lower().lstrip(".")

        return documents


    @abstractmethod
    def _load(self, path: Path):
        """Extract documents using a format-specific implementation."""
        return NotImplementedError
    


        

In [ ]:
from pathlib import Path


class PDFLoader(BaseDocumentLoader):
    """Load PDF files, preserving page metadata."""

    supported_extensions = {".pdf"}

    def _load(self, path: Path) -> List[Document]:
        return PyPDFLoader(str(path)).load()



class DOCXLoader(BaseDocumentLoader):
    """Load Microsoft Word DOCX files"""
    supported_extensions = {".docx"}

    def _load(self, path: Path) -> List[Document]:
        return Docx2txtLoader(str(path)).load()



def load_document(file_path: str | Path) -> List[Document]:
    """select the correct loader based on the file extensions"""
    path = Path(file_path)
    extension = path.suffix.lower()

    loaders = {
        ".pdf": PDFLoader,
        ".docx": DOCXLoader
    }

    loader_class = loaders.get(extension)
    if loader_class is None:
        raise ValueError(
            f"Unsupported file type: {extension}"
            f"Supported file formats: .pdf, .docx"
        )

    return loader_class().load(path)
